## Part 1 — Setup

### 1.1 Install libraries
On Colab just run the cell. For a local setup use `pip install -r requirements.txt` instead.

In [ ]:
%pip install -q openai pdfplumber sentence-transformers langchain-text-splitters rank_bm25 faiss-cpu qdrant-client pinecone pandas matplotlib

### 1.2 Get the sample data
We use `Basic_Home_Remedies.pdf` (text) and `sales_report.csv` (a table) from the course repository.

In [ ]:
import os

if not os.path.exists("deep_learning"):
    !git clone --depth 1 https://github.com/deepanrajm/deep_learning.git

PDF_PATH = "deep_learning/RAG/Basic_Home_Remedies.pdf"

### 1.3 API keys
- **OpenRouter** (LLM): create a free key at https://openrouter.ai/settings/keys
- **Pinecone** (only for Part 7.3): create a free key at https://app.pinecone.io

**On Colab:** click the 🔑 *Secrets* icon in the left sidebar, add `OPENROUTER_API_KEY` (and `PINECONE_API_KEY`), and switch on *Notebook access*.<br>
**Locally:** set them as environment variables, or just type them when the cell asks.

### 1.4 Connect to the LLM
OpenRouter speaks the same API as OpenAI, so we use the plain `openai` client — no extra framework needed.
To use a model running locally in **LM Studio**, switch to Option B.

In [ ]:
from openai import OpenAI

# Option A: OpenRouter (cloud)
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key="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")

# Option B: LM Studio (local) - start the LM Studio server, then uncomment:
# client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

LLM_MODEL = "nvidia/nemotron-3.5-lightning:free"   # any chat model from https://openrouter.ai/models
VISION_MODEL = "google/gemma-4-31b-it:free"        # must accept images (used in Part 12)


def ask_llm(prompt, model=LLM_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content


print(ask_llm("In one sentence, what is Retrieval-Augmented Generation?"))

## Part 2 — Load the document
`pdfplumber` reads the text of each page. We join the pages into one long string.

In [ ]:
import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:
    pages = [page.extract_text() for page in pdf.pages]

text = "\n".join(pages)
text = text.replace("\uf0b7", "-")   # this PDF uses a special bullet symbol; make it a normal dash

print("Pages:", len(pages), "| Characters:", len(text))
print("-" * 60)
print(text[:600])

## Part 3 — Embeddings: turning text into vectors
An **embedding model** converts text into a list of numbers (a vector).
Texts with similar *meaning* get vectors that point in similar directions.

We normalise the vectors to length 1, so the **dot product = cosine similarity**
(1.0 = same meaning, around 0 = unrelated).

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, 384 numbers per text


def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True)


vectors = embed([
    "remedy for cough",
    "honey and lemon soothe a sore throat",
    "regular exercise keeps the heart healthy",
])

print("Each text becomes", vectors.shape[1], "numbers. First 8:", vectors[0][:8].round(3))
print("cough  vs  honey/throat :", round(float(vectors[0] @ vectors[1]), 3))
print("cough  vs  exercise     :", round(float(vectors[0] @ vectors[2]), 3))

## Part 4 — Chunking strategies
LLMs and embedding models work best on small pieces of text, so we split the document into **chunks**.
How we split changes what the retriever can find.

| Strategy | Idea | Good for |
|---|---|---|
| Fixed-size | Cut every N characters (with overlap) | Simple baseline |
| Recursive | Split on paragraphs → lines → sentences → words until small enough | Most general documents |
| Section-based | Split on the document's own headings | Well-structured docs (manuals, guides) |
| Semantic | Start a new chunk when the meaning changes | Documents without clear structure |

Each chunk is stored as a small dictionary: `{"id", "text", "type"}`.

In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter


def to_chunks(pieces, chunk_type="text"):
    pieces = [p.strip() for p in pieces if p.strip()]
    return [{"id": i, "text": p, "type": chunk_type} for i, p in enumerate(pieces)]


# 1) Fixed-size: cut every `size` characters, repeating `overlap` characters
def fixed_size_chunks(text, size=300, overlap=50):
    return [text[start:start + size] for start in range(0, len(text), size - overlap)]


# 2) Recursive: LangChain tries "\n\n", then "\n", then " " ... until pieces fit
def recursive_chunks(text, size=300, overlap=50):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    return splitter.split_text(text)


# 3) Section-based: split before every heading like "2.1 Cold & Flu" or "3. Healthy Lifestyle"
def section_chunks(text, min_length=50):
    sections = re.split(r"\n(?=\d+\.\d* )", text)
    return [s for s in sections if len(s) >= min_length]    # drop heading-only pieces


# 4) Semantic: compare each sentence with the previous one; low similarity = new topic
def semantic_chunks(text, threshold=0.3):
    sentences = re.split(r"(?<=[.!?])\s+", text.replace("\n", " "))
    sentence_vectors = embed(sentences)
    pieces, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        similarity = float(sentence_vectors[i - 1] @ sentence_vectors[i])
        if similarity < threshold:
            pieces.append(" ".join(current))
            current = []
        current.append(sentences[i])
    pieces.append(" ".join(current))
    return pieces


strategies = {
    "Fixed": to_chunks(fixed_size_chunks(text)),
    "Recursive": to_chunks(recursive_chunks(text)),
    "Section": to_chunks(section_chunks(text)),
    "Semantic": to_chunks(semantic_chunks(text)),
}

for name, chunk_list in strategies.items():
    lengths = [len(c["text"]) for c in chunk_list]
    print(f"{name:10} chunks={len(chunk_list):3}   avg={sum(lengths) // len(lengths):4}   "
          f"min={min(lengths):4}   max={max(lengths):4} characters")

In [ ]:
# Look at the first two chunks from each strategy.
# Notice how "Fixed" cuts words in half, while "Section" keeps each topic together.
for name, chunk_list in strategies.items():
    print(f"\n==================== {name} ====================")
    for c in chunk_list[:2]:
        print(f"[chunk {c['id']}] {c['text'][:250]}\n")

For the rest of the notebook we use **section-based** chunks, because this guide has clear headings.
👉 *Exercise:* change `"Section"` to `"Recursive"` or `"Semantic"` and re-run everything below to see how answers change.

In [8]:
chunks = strategies["Section"]
print("Using", len(chunks), "chunks")

Using 16 chunks


## Part 5 — Vector search from scratch
A vector database does two jobs:
1. **Store** a vector for every chunk.
2. **Search**: embed the question and return the chunks whose vectors are closest.

Here is that idea in ~10 lines of NumPy — real vector DBs do the same thing, just faster and at scale.

In [ ]:
class SimpleVectorStore:
    # A tiny in-memory vector database

    def __init__(self, chunks):
        self.chunks = chunks
        self.vectors = embed([c["text"] for c in chunks])      # one vector per chunk

    def search(self, query, k=3):
        query_vector = embed([query])[0]
        scores = self.vectors @ query_vector                    # cosine similarity with every chunk
        best = np.argsort(scores)[::-1][:k]                     # indices of the k highest scores
        return [(self.chunks[i], float(scores[i])) for i in best]


def show_results(results, title="Results"):
    print(f"\n--- {title} ---")
    for rank, (chunk, score) in enumerate(results, start=1):
        preview = chunk["text"][:110].replace("\n", " ")
        print(f"{rank}. [{chunk['type']}] score={score:.3f} | {preview}...")


store = SimpleVectorStore(chunks)
chunk_vectors = store.vectors      # we reuse these vectors for FAISS, Qdrant and Pinecone later

show_results(store.search("remedy for cough"), "Vector search: 'remedy for cough'")
show_results(store.search("I can't sleep at night"), "Vector search: 'I can't sleep at night'")

## Part 6 — Basic RAG: Retrieve ➜ Augment ➜ Generate
1. **Retrieve** the top-k chunks for the question
2. **Augment**: paste them into the prompt as context
3. **Generate**: the LLM answers using only that context

`rag()` accepts any search function, so we can plug in every retriever we build later.

In [ ]:
PROMPT = '''You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".
Keep the answer short (3-4 sentences).

Context:
{context}

Question: {question}
Answer:'''


def rag(question, search_fn, k=3):
    results = search_fn(question, k)                                   # 1. Retrieve
    show_results(results, "Retrieved context")
    context = "\n\n".join(chunk["text"] for chunk, _ in results)       # 2. Augment
    answer = ask_llm(PROMPT.format(context=context, question=question))  # 3. Generate
    print("\nAnswer:", answer)
    return answer


rag("What is a good home remedy for cough?", store.search)

In [ ]:
# The "I don't know" rule in the prompt stops the LLM from making things up:
rag("What is the capital of France?", store.search)

## Part 7 — Vector stores & vector databases
Our `SimpleVectorStore` compares the question with **every** chunk and forgets everything when Python stops.
Real tools add speed, persistence, metadata filtering and scaling.

| Tool | Type | Setup | Data survives restart? |
|---|---|---|---|
| **FAISS** | Library (vector *store*) | `pip install` only | No (unless you save the file) |
| **Qdrant** | Vector *database* (open source) | In-memory / Docker / Cloud | Yes (Docker / Cloud) |
| **Pinecone** | Vector *database* (fully managed cloud) | Free account + API key | Yes |

All three below use the **same chunk vectors**, so they should return the same top results.

### 7.1 FAISS (Facebook AI Similarity Search)

In [ ]:
import faiss

faiss_index = faiss.IndexFlatIP(chunk_vectors.shape[1])   # IP = inner product (= cosine for normalised vectors)
faiss_index.add(chunk_vectors)
print("Vectors in FAISS:", faiss_index.ntotal)


def faiss_search(query, k=3):
    scores, ids = faiss_index.search(embed([query]), k)
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0])]


show_results(faiss_search("remedy for cough"), "FAISS")

### 7.2 Qdrant
Pick **one** way to run Qdrant:

**Option 1 — In-memory (no setup, default below).** Runs inside Python; perfect for class.

**Option 2 — Docker on your laptop (local setup).**
```bash
docker run -p 6333:6333 -v $(pwd)/qdrant_data:/qdrant/storage qdrant/qdrant
```
Dashboard: http://localhost:6333/dashboard — then use `QdrantClient(url="http://localhost:6333")`.
(Docker does not run inside Colab; use Option 1 or 3 there.)

**Option 3 — Qdrant Cloud (free tier).**
1. Sign up at https://cloud.qdrant.io and create a free cluster.
2. Copy the cluster URL and create an API key.
3. Add `QDRANT_API_KEY` to Colab Secrets and use the Option 3 line below.

Qdrant words: a **collection** = a table, a **point** = one row (`id` + `vector` + `payload`). The payload is our chunk dictionary (metadata).

In [ ]:
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(":memory:")                                              # Option 1
# qdrant = QdrantClient(url="http://localhost:6333")                           # Option 2
# qdrant = QdrantClient(url="https://YOUR-CLUSTER-URL", api_key=get_secret("QDRANT_API_KEY"))  # Option 3

COLLECTION = "home_remedies"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=chunk_vectors.shape[1], distance=models.Distance.COSINE),
)

qdrant.upsert(
    collection_name=COLLECTION,
    points=[
        models.PointStruct(id=c["id"], vector=v.tolist(), payload=c)
        for c, v in zip(chunks, chunk_vectors)
    ],
)
print("Points in Qdrant:", qdrant.count(COLLECTION).count)


def qdrant_search(query, k=3):
    hits = qdrant.query_points(COLLECTION, query=embed([query])[0].tolist(), limit=k).points
    return [(hit.payload, hit.score) for hit in hits]


show_results(qdrant_search("remedy for cough"), "Qdrant")

### 7.3 Pinecone
Setup (one time, ~2 minutes):
1. Sign up at https://app.pinecone.io (the free *Starter* plan is enough).
2. Go to **API Keys** → copy the default key.
3. Add it as `PINECONE_API_KEY` in Colab Secrets (or an environment variable locally).

No key yet? Skip this section — everything else works without it.

The code below creates a **serverless index** (Pinecone's word for a collection) if it doesn't exist yet.

In [ ]:
import time
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=get_secret("PINECONE_API_KEY"))
INDEX_NAME = "rag-demo"

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=chunk_vectors.shape[1],
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),   # region available on the free plan
    )

pinecone_index = pc.Index(INDEX_NAME)

pinecone_index.upsert(vectors=[
    {"id": str(c["id"]), "values": v.tolist(), "metadata": {"text": c["text"], "type": c["type"]}}
    for c, v in zip(chunks, chunk_vectors)
])
time.sleep(10)   # Pinecone needs a few seconds before new vectors become searchable


def pinecone_search(query, k=3):
    response = pinecone_index.query(vector=embed([query])[0].tolist(), top_k=k, include_metadata=True)
    return [(match.metadata, match.score) for match in response.matches]


show_results(pinecone_search("remedy for cough"), "Pinecone")

# When you're done: pc.delete_index(INDEX_NAME)

In [ ]:
# Same RAG function, different vector database - only the search function changes
rag("How can I relieve a headache naturally?", qdrant_search)

## Part 8 — Keyword search with BM25
Vector search matches **meaning**. **BM25** is the classic search-engine algorithm that matches **exact words**:
- a word that appears often in a chunk → higher score
- a rare word (like *"Epsom"*) counts much more than a common word (like *"the"*)

It needs no embedding model at all.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize(text):
    return re.findall(r"\w+", text.lower())    # "Epsom Salt Bath!" -> ["epsom", "salt", "bath"]


bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])


def bm25_search(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    best = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in best]


# Exact rare keyword -> BM25 is great
show_results(bm25_search("Epsom salt"), "BM25: 'Epsom salt'")
show_results(store.search("Epsom salt"), "Vector: 'Epsom salt'")

# Same meaning, different words -> vector search is great, BM25 struggles
show_results(bm25_search("my throat feels scratchy"), "BM25: 'my throat feels scratchy'")
show_results(store.search("my throat feels scratchy"), "Vector: 'my throat feels scratchy'")

## Part 9 — Hybrid search: best of both worlds
Combine keyword (BM25) and meaning (vector) results.

**A) Reciprocal Rank Fusion (RRF)** — uses only the *rank* of each chunk in each list:
`score = 1/(60 + rank_in_vector_list) + 1/(60 + rank_in_bm25_list)`.
No need to compare BM25 scores with cosine scores. This is the most common choice.

**B) Weighted fusion** — scale both scores to 0..1, then mix them:
`score = alpha × vector_score + (1 − alpha) × bm25_score`.
`alpha = 1` → pure vector, `alpha = 0` → pure BM25.

In [ ]:
def hybrid_rrf_search(query, k=3, candidates=10):
    fused = {}
    for results in [store.search(query, candidates), bm25_search(query, candidates)]:
        for rank, (chunk, _) in enumerate(results, start=1):
            fused[chunk["id"]] = fused.get(chunk["id"], 0) + 1 / (60 + rank)
    best_ids = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(chunks[i], fused[i]) for i in best_ids]


def hybrid_weighted_search(query, k=3, alpha=0.5):
    vector_scores = store.vectors @ embed([query])[0]
    bm25_scores = bm25.get_scores(tokenize(query))

    def scale(x):                       # min-max scaling to the 0..1 range
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    combined = alpha * scale(vector_scores) + (1 - alpha) * scale(bm25_scores)
    best = np.argsort(combined)[::-1][:k]
    return [(chunks[i], float(combined[i])) for i in best]


query = "turmeric for joint pain"
show_results(store.search(query), "Vector only")
show_results(bm25_search(query), "BM25 only")
show_results(hybrid_rrf_search(query), "Hybrid (RRF)")
show_results(hybrid_weighted_search(query, alpha=0.5), "Hybrid (weighted, alpha=0.5)")

## Part 10 — Re-ranking
Fast retrievers are good at finding *candidates*, not at picking the *best* ones.
So we use a 2-stage pipeline:

```
Question ─► fast retriever (top 10) ─► slow but smart re-ranker ─► best 3 ─► LLM
```

**A) Cross-encoder re-ranker** — reads the question and a chunk *together* and outputs a relevance score.
More accurate than comparing two separate embeddings, but too slow to run on every chunk — perfect for 10 candidates.

**B) LLM re-ranker** — simply ask the LLM to rate each chunk from 0 to 10. Easy to understand, but slower and costs API calls.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query, results, top_n=3):
    pairs = [(query, chunk["text"]) for chunk, _ in results]
    scores = reranker.predict(pairs)
    best = np.argsort(scores)[::-1][:top_n]
    return [(results[i][0], float(scores[i])) for i in best]


query = "what should I drink to help digestion?"
candidates = store.search(query, k=10)
show_results(candidates[:3], "Before re-ranking (vector top 3)")
show_results(rerank(query, candidates), "After cross-encoder re-ranking")

In [17]:
def llm_rerank(query, results, top_n=3):
    scored = []
    for chunk, _ in results:
        reply = ask_llm(
            "On a scale of 0 to 10, how useful is this passage for answering the question?\n"
            f"Question: {query}\nPassage: {chunk['text']}\n"
            "Reply with only the number."
        )
        numbers = re.findall(r"\d+", reply)
        scored.append((chunk, float(numbers[0]) if numbers else 0.0))
    return sorted(scored, key=lambda pair: pair[1], reverse=True)[:top_n]


show_results(llm_rerank(query, candidates[:5]), "After LLM re-ranking (top 5 candidates)")


--- After LLM re-ranking (top 5 candidates) ---
1. [text] score=8.000 | 2.4 Indigestion & Acidity - Fennel Seeds: Chew fennel seeds after meals to aid digestion. - Baking Soda: Mix 1...
2. [text] score=7.000 | 2.5 Constipation - Warm Lemon Water: Drink on an empty stomach to stimulate digestion. - Flaxseeds: Soak overn...
3. [text] score=3.000 | 3.1 Balanced Diet - Eat whole foods, including fruits, vegetables, whole grains, and lean proteins. - Reduce p...


### The full advanced pipeline: hybrid search ➜ cross-encoder re-rank ➜ LLM

In [18]:
def advanced_search(query, k=3):
    candidates = hybrid_rrf_search(query, k=10)
    return rerank(query, candidates, top_n=k)


rag("What can I do for acidity after meals?", advanced_search)


--- Retrieved context ---
1. [text] score=7.238 | 2.4 Indigestion & Acidity - Fennel Seeds: Chew fennel seeds after meals to aid digestion. - Baking Soda: Mix 1...
2. [text] score=-8.322 | 3.1 Balanced Diet - Eat whole foods, including fruits, vegetables, whole grains, and lean proteins. - Reduce p...
3. [text] score=-9.146 | 3.5 Detox & Immunity Boosting - Start the day with warm lemon water for detoxification. - Consume garlic and g...

Answer: You can chew fennel seeds after meals to aid digestion. For instant relief from acidity, mix 1/2 tsp baking soda in warm water and drink. These are the remedies suggested in the context for acidity after meals.


'You can chew fennel seeds after meals to aid digestion. For instant relief from acidity, mix 1/2 tsp baking soda in warm water and drink. These are the remedies suggested in the context for acidity after meals.'

## Part 11 — Compare all retrievers side by side
Same question, every retriever, top result only. Try your own questions!

In [ ]:
retrievers = {
    "Vector (NumPy)": store.search,
    "FAISS": faiss_search,
    "Qdrant": qdrant_search,
    "Pinecone": pinecone_search,
    "BM25": bm25_search,
    "Hybrid RRF": hybrid_rrf_search,
    "Hybrid weighted": hybrid_weighted_search,
    "Hybrid + rerank": advanced_search,
}

query = "how to get better sleep"
print("Question:", query, "\n")
for name, search_fn in retrievers.items():
    chunk, score = search_fn(query, 1)[0]
    print(f"{name:16} score={score:7.3f} | {chunk['text'][:80].replace(chr(10), ' ')}...")